# Step 13 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew`. You'll run the same two agents under two different `Process` strategies: `sequential` — the same fixed pipeline this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files — and `hierarchical`, where a manager decides at runtime which agent handles each task, instead of that being fixed in code.

## Learning objective

By the end of this notebook, you will:

- Understand why a second, complementary agent can produce something neither agent produces alone
- Have chained two `Task`s together with `context=[...]`, so one agent's output feeds directly into another's input
- Have run your first real `Crew` with two agents and `process=Process.sequential`
- Understand how `process=Process.hierarchical` differs: task order is still fixed, but *which* agent handles each task is a manager's runtime decision, not something wired into the code
- Have run the same two agents under a manager, via `manager_llm` (or a custom `manager_agent`)

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- The same `.env` setup as the previous steps

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

![AutoGen: conversable agents solving tasks in joint or hierarchical patterns](../assets/autogen-wu2023-fig1.png)
*Figure 1 from Wu et al. (2023). Reproduced for educational use in this course.*

## How this works

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential`:

1. **`researcher`** is Step 09's exact same Researcher agent, unchanged.
2. **`analyst`** is a new agent with a different `role`/`goal`/`backstory` — someone who turns raw findings into a report for a specific audience, a job the Researcher was never asked to do.
3. **`research_task`** is assigned to the Researcher; **`analysis_task`** is assigned to the Analyst, with one crucial addition: `context=[research_task]`. This tells CrewAI to feed the first task's output into the second automatically — the same idea as Step 06's chain prompting, but wired declaratively instead of copy-pasted by hand.
4. **`crew.kickoff()`** runs both tasks in the order they appear in `tasks=[...]`, because `process=Process.sequential`.

Everything else in the cell (the `tracing=True` flag and the `crewai_event_bus.flush()` call afterward) is optional bookkeeping that prints a shareable trace link — it doesn't affect what the agents do.

Further down, the same two agents run again under `process=Process.hierarchical` — a different way of wiring the same pieces together.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── Crew — same sequential process as src/research_crew/crew.py ──────────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d5a49996-44b6-4b32-bdb8-d80ecf254e32                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 3a9135bb-e41e-47fe-b2c7-6352b5ce87ab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the            │
│  risk-based regulatory framework. The Act employs a tiered approach, calibrating the stringency of obligations  │
│  based on the potential harm an AI system poses to the safety, health, and fundamental rights of EU citizens.   │
│                                                                                                                 │
│  ### I. The Four Risk-Based Categories                                                                          │
│                                                                                                                 │
│  1.  **Unacceptable Risk (Prohibited):**                                                                        │
│      These systems are considered a clear threat to fundamental rights. Their development, deployment, and use  │
│  are strictly banned within the EU.                                                                             │
│      *   *Examples:* AI-driven social scoring by governments, biometric categorization systems using sensitive  │
│  data (e.g., political or religious beliefs), and AI that uses subliminal techniques to manipulate behavior to  │
│  cause harm.                                                                                                    │
│                                                                                                                 │
│  2.  **High-Risk AI Systems:**                                                                                  │
│      Systems that pose significant risks to the health, safety, or fundamental rights of natural persons.       │
│  These are subject to strict compliance requirements before entering the market.                                │
│      *   *Examples:* AI used in critical infrastructure, education and vocational training (e.g., assessing     │
│  exams), employment (e.g., CV-sorting software), essential private and public services (e.g., credit scoring),  │
│  and law enforcement.                                                                                           │
│                                                                                                                 │
│  3.  **Limited Risk (Transparency Obligations):**                                                               │
│      AI systems that carry a specific risk of manipulation or lack of transparency. Providers must ensure       │
│  users are aware they are interacting with an AI.                                                               │
│      *   *Examples:* Chatbots, AI systems that generate or manipulate image, audio, or video content (e.g.,     │
│  deepfakes).                                                                                                    │
│                                                                                                                 │
│  4.  **Minimal/No Risk:**                                                                                       │
│      The vast majority of AI systems currently in use in the EU (e.g., spam filters, AI-enabled video games).   │
│  These face no additional obligations beyond existing consumer protection and data laws, though the Act         │
│  encourages voluntary codes of conduct.                

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what obligations     │
│  apply to providers of high-risk AI systems.                                                                    │
│  Agent:                                                                                                         │
│  EU AI Act Senior Data Researcher                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Reporting Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Compliance Report: EU AI Act Obligations for AI-Powered Hiring Tools                                       │
│                                                                                                                 │
│  **To:** Product Development Team                                                                               │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Subject:** Regulatory Requirements for AI-Powered Hiring Systems                                             │
│                                                                                                                 │
│  #### 1. Classification & Scope                                                                                 │
│  Under the EU AI Act, AI systems intended to be used for the recruitment or selection of natural persons,       │
│  including advertising vacancies, screening or filtering applications, and evaluating candidates, are           │
│  explicitly classified as **High-Risk AI Systems**.                                                             │
│                                                                                                                 │
│  Because our tool impacts fundamental rights (access to employment), we are subject to the most stringent set   │
│  of requirements under the Act. Compliance is not optional; failure to adhere risks fines of up to **€35        │
│  million or 7% of total worldwide annual turnover.**                                                            │
│                                                                                                                 │
│  #### 2. Mandatory Compliance Obligations                                                                       │
│  We must integrate the following requirements into our development and operational lifecycle:                   │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous process to identify and mitigate foreseeable risks      │
│  throughout the system's entire lifecycle.                                                                      │
│  *   **Data Governance:** Ensure training, validation, and testing datasets are representative, relevant, and   │
│  free of bias. We must implement rigorous testing to detect and mitigate discriminatory patterns.               │
│  *   **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  readily available for national competent authorities.                                                          │
│  *   **Automated Logging:** The system must automatically generate logs to ensure traceability of               │
│  decision-making and system performance.                                                                        │
│  *   **Transparency:** Users must be able to interpret the system’s output. We must provide clear, concise      │
│  instructions for use.                                                                                          │
│  *   **Human Oversight:** The system must be designed to allow human operators to effectively oversee,          │
│  intervene, override, or shut down the AI’s processes. 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  EU AI Act Reporting Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 4f5ea7ec-b869-401b-a1d5-e1abb09a0c4f                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/4f5ea7ec-b869-401b-a1d5-e1abb09a0c4f?access_code=TRA │
│ CE-939b6122d7                                                                                                   │
│ 🔑 Access Code: TRACE-939b6122d7                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d5a49996-44b6-4b32-bdb8-d80ecf254e32                                                                           │
│  Final Output: ### Compliance Report: EU AI Act Obligations for AI-Powered Hiring Tools                         │
│                                                                                                                 │
│  **To:** Product Development Team                                                                               │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Subject:** Regulatory Requirements for AI-Powered Hiring Systems                                             │
│                                                                                                                 │
│  #### 1. Classification & Scope                                                                                 │
│  Under the EU AI Act, AI systems intended to be used for the recruitment or selection of natural persons,       │
│  including advertising vacancies, screening or filtering applications, and evaluating candidates, are           │
│  explicitly classified as **High-Risk AI Systems**.                                                             │
│                                                                                                                 │
│  Because our tool impacts fundamental rights (access to employment), we are subject to the most stringent set   │
│  of requirements under the Act. Compliance is not optional; failure to adhere risks fines of up to **€35        │
│  million or 7% of total worldwide annual turnover.**                                                            │
│                                                                                                                 │
│  #### 2. Mandatory Compliance Obligations                                                                       │
│  We must integrate the following requirements into our development and operational lifecycle:                   │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous process to identify and mitigate foreseeable risks      │
│  throughout the system's entire lifecycle.                                                                      │
│  *   **Data Governance:** Ensure training, validation, and testing datasets are representative, relevant, and   │
│  free of bias. We must implement rigorous testing to detect and mitigate discriminatory patterns.               │
│  *   **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  readily available for national competent authorities.                                                          │
│  *   **Automated Logging:** The system must automatically generate logs to ensure traceability of               │
│  decision-making and system performance.                                                                        │
│  *   **Transparency:** Users must be able to interpret the system’s output. We must provide clear, concise      │
│  instructions for use.                                

=== Researcher output ===
As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the risk-based regulatory framework. The Act employs a tiered approach, calibrating the stringency of obligations based on the potential harm an AI system poses to the safety, health, and fundamental rights of EU citizens.

### I. The Four Risk-Based Categories

1.  **Unacceptable Risk (Prohibited):**
    These systems are considered a clear threat to fundamental rights. Their development, deployment, and use are strictly banned within the EU.
    *   *Examples:* AI-driven social scoring by governments, biometric categorization systems using sensitive data (e.g., political or religious beliefs), and AI that uses subliminal techniques to manipulate behavior to cause harm.

2.  **High-Risk AI Systems:**
    Systems that pose significant risks to the health, safety, or fundamental rights of natural persons. These are subject to strict compliance requirements before ent

### A non-sequential pattern: manager delegation (`Process.hierarchical`)

Above, each `Task` is hard-wired to one `Agent` via `agent=...`, and `crew.kickoff()` runs them in the order they appear in `tasks=[...]`. `Process.hierarchical` removes that fixed wiring: the tasks below don't get an `agent=` at all. Instead, a **manager** — either an `Agent` CrewAI builds for you from `manager_llm`, or your own `Agent` passed as `manager_agent` — reads each task and decides, at that moment, which of the crew's agents is best suited, then delegates to them via a tool call.

Two things worth being precise about, since "hierarchical" invites the wrong mental model:

- **Task order still isn't dynamic.** The `for` loop over `tasks=[...]` doesn't change — task 1 still runs before task 2. What's dynamic is *who* does each task, decided fresh every run, not *when* it happens.
- **The manager is not one of your agents.** It's a separate role that does no task work itself — it only reads tasks and delegates. CrewAI enforces this: a `manager_agent` may not also appear in `agents=[...]`, and it may not be given `tools=` (only the agents it delegates to can have tools).

The `researcher` and `analyst` from above are reused unchanged below — same `role`/`goal`/`backstory`, no code changes to either agent. Only the `Crew`'s wiring is different.

In [2]:
# ── Same two tasks, but neither gets an `agent=` — the manager decides who does what ──
research_task_h = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
)

analysis_task_h = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
)

# ── Crew — same two agents as coworkers, orchestrated by a manager instead of fixed code ──
hierarchical_crew = Crew(
    agents=[researcher, analyst],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task_h, analysis_task_h],
    process=Process.hierarchical,
    manager_llm=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"),
    verbose=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result_h = hierarchical_crew.kickoff()

print("=== Research task, as delegated by the manager ===")
print(research_task_h.output.raw)
print("\n=== Analysis task, as delegated by the manager ===")
print(analysis_task_h.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  90b4da90-9874-4a4a-99c6-e767d14940b1                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: d0180c8f-9989-4336-9da8-b2933405195d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': "The user requires a comprehensive and structured explanation of the EU AI Act's risk-based  │
│  categories (Unacceptable, High-Risk, Limited, Minimal) and a detailed breakdown of the specific ...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Create a detailed and structured explanation of the EU AI Act's four risk categories and the specific    │
│  regulatory obligations for providers of high-risk AI systems.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Compliance Report: EU AI Act Requirements for AI-Powered Hiring Tool                                         │
│                                                                                                                 │
│  **To:** Product and Engineering Teams                                                                          │
│  **Subject:** Mandatory Compliance Roadmap for EU Market Access                                                 │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, our AI-powered hiring/CV screening tool is explicitly classified as a **High-Risk AI      │
│  System**. This classification is due to its function in employment-related decision-making, which directly     │
│  impacts fundamental rights. We are legally required to adhere to the stringent compliance framework outlined   │
│  in the Act.                                                                                                    │
│                                                                                                                 │
│  ### 2. Mandatory Obligations                                                                                   │
│  To ensure our product remains legally compliant and viable for the EU market, we must integrate the following  │
│  nine pillars into our development lifecycle:                                                                   │
│                                                                                                                 │
│  1.  **Risk Management System:** Implement a continuous, iterative process to identify and mitigate             │
│  foreseeable risks throughout the system's lifecycle.                                                           │
│  2.  **Data & Governance:** Establish strict governance for training and testing datasets. We must ensure data  │
│  is representative and proactively tested for biases that could lead to discriminatory hiring practices.        │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating conformity with all legal  │
│  requirements, ready for inspection by national authorities.                                                    │
│  4.  **Automatic Logging:** Enable persistent, automated logging of events to ensure traceability and           │
│  auditability.                                                                                                  │
│  5.  **Transparency & Information:** Provide clear, accessible instructions to users regarding the system's     │
│  capabilities, limitations, and operational requirements.                                                       │
│  6.  **Human Oversight:** Engineer the system to ensure meaningful "human-in-the-loop" capabilities, allowing   │
│  operators to override or halt system decisions.                                                                │
│  7.  **Accuracy, Robustness & Cybersecurity:** Ensure high performance standards and implement strong defenses  │
│  against cyber-attacks (e.g., data poisoning).                                                                  │
│  8.  **Quality Management System (QMS):** Formalize a Q

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  90b4da90-9874-4a4a-99c6-e767d14940b1                                                                           │
│  Final Output: # Compliance Report: EU AI Act Requirements for AI-Powered Hiring Tool                           │
│                                                                                                                 │
│  **To:** Product and Engineering Teams                                                                          │
│  **Subject:** Mandatory Compliance Roadmap for EU Market Access                                                 │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, our AI-powered hiring/CV screening tool is explicitly classified as a **High-Risk AI      │
│  System**. This classification is due to its function in employment-related decision-making, which directly     │
│  impacts fundamental rights. We are legally required to adhere to the stringent compliance framework outlined   │
│  in the Act.                                                                                                    │
│                                                                                                                 │
│  ### 2. Mandatory Obligations                                                                                   │
│  To ensure our product remains legally compliant and viable for the EU market, we must integrate the following  │
│  nine pillars into our development lifecycle:                                                                   │
│                                                                                                                 │
│  1.  **Risk Management System:** Implement a continuous, iterative process to identify and mitigate             │
│  foreseeable risks throughout the system's lifecycle.                                                           │
│  2.  **Data & Governance:** Establish strict governance for training and testing datasets. We must ensure data  │
│  is representative and proactively tested for biases that could lead to discriminatory hiring practices.        │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating conformity with all legal  │
│  requirements, ready for inspection by national authorities.                                                    │
│  4.  **Automatic Logging:** Enable persistent, automated logging of events to ensure traceability and           │
│  auditability.                                                                                                  │
│  5.  **Transparency & Information:** Provide clear, accessible instructions to users regarding the system's     │
│  capabilities, limitations, and operational requirements.                                                       │
│  6.  **Human Oversight:** Engineer the system to ensure meaningful "human-in-the-loop" capabilities, allowing   │
│  operators to override or halt system decisions.                                                                │
│  7.  **Accuracy, Robustness & Cybersecurity:** Ensure 

=== Research task, as delegated by the manager ===


# Understanding the EU AI Act: Risk Categorization and Compliance Obligations

The EU AI Act classifies artificial intelligence systems into four distinct risk tiers. This risk-based approach ensures that regulatory burdens are proportionate to the potential threats posed to fundamental rights, safety, and the rule of law.

## Part I: The Four Risk-Based Categories

### 1. Unacceptable Risk (Prohibited AI)
Systems deemed to pose a clear and present danger to the fundamental rights of EU citizens are prohibited. Their development and deployment are strictly forbidden.
*   **Key Prohibitions:** 
    *   AI systems that deploy manipulative or deceptive techniques to distort human behavior.
    *   Systems that exploit vulnerabilities of specific groups (age, disability, social situation).
    *   "Social scoring" systems conducted by governments.
    *   Real-time remote biometric identification in publicly accessible spaces for law enforcement (with very narrow exceptions).

### 2. High

## Your task

1. Run the sequential cell. Compare the Analyst's report to Step 09's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Run the hierarchical cell. With `verbose=True`, the console output shows the manager's own reasoning and its "Delegate work to coworker" / "Ask question to coworker" tool calls — find the line where it picks the Researcher for the first task and the Analyst for the second. Compare `result_h` to `result` from the sequential run: same two jobs, same two agents — did the manager's routing produce a meaningfully different outcome, or basically the same one with extra steps (and extra LLM calls) in between?

4. Now swap in your own team's topic — keep the Researcher agent from Step 09, and give it a second, genuinely complementary role and task suited to your topic.

5. This is where your project shifts from guided exercises to your own design: use everything from Steps 03–13 to build and document your own agent. Fill in `EVALUATION.md` — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full report structure and what's expected.

## Shortcomings

The sequential pipeline is fixed: the Researcher always runs first, the Analyst always runs second, and the order never changes based on what either agent actually finds. `Process.hierarchical` fixes *who* handles each task, not *when* — the task order in `tasks=[...]` is still fixed, and it costs more: every task now takes an extra LLM call (the manager's own reasoning) on top of whichever agent it delegates to, and the manager's choice of delegate isn't guaranteed to be consistent between runs. For two agents with clearly non-overlapping roles like these, that extra machinery mostly buys you nothing — it starts to pay off once you have enough agents, or similar-enough roles, that hard-coding `agent=...` on every task stops being obvious.

Neither `Crew` above uses any of the tools/MCP/RAG grounding from Steps 10–12 — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

This is the last step in the exercise series. From here, the main [README's use-case table](../../README.md#use-cases-to-pick-from) and `EVALUATION.md` ask you to step back and judge, for your own topic, which combination of everything covered — single vs. multi-agent, sequential vs. hierarchical, tools, MCP, RAG — is actually worth the added complexity.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered briefly in Step 02

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. The hierarchical crew is the more natural fit for this: add the agent to `agents=[...]` and a `Task` for it to `tasks=[...]`, but don't pin it with `agent=...` — let the manager decide on its own whether and when to bring it in, rather than wiring it into a fixed spot in the pipeline like you'd have to for the sequential version. Does the output meaningfully change? If it doesn't, ask yourself why.

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.